# Раздел 9 · Мультивекторы и Late Interaction (ColBERT)

> Этот ноутбук продолжает Раздел 3 («Коллекции и точки») — использует уже созданную там коллекцию `BOOKS_COLLECTION`. Если ниже вылетит `AssertionError: Коллекция ... ещё не создана` — сначала откройте и выполните ноутбук Раздела 3, потом возвращайтесь сюда.

In [ ]:
%pip install qdrant-client sentence-transformers ipykernel fastembed --quiet


> Если сразу после установки следующая ячейка падает с `ModuleNotFoundError` на только что установленном пакете — перезапустите ядро (Kernel → Restart) и выполните ячейки заново с начала: иногда ядро не видит пакет, установленный в уже запущенном процессе, без перезапуска.

In [ ]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    MultiVectorConfig, MultiVectorComparator, Prefetch
)

print("OK: библиотеки импортированы")


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [ ]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("../books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


In [ ]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


## Мультивекторы и Late Interaction / ColBERT

Здесь снова нужна **новая** коллекция, как и в разделе про sparse-векторы и гибридный поиск:
мультивекторный тип (`multivector_config`) должен быть объявлен на именованном векторе с самого
создания коллекции, задним числом в `BOOKS_COLLECTION` его не добавить. Зато dense-вектор для
новой коллекции переиспользуем из уже посчитанного в `BOOKS_COLLECTION` (раздел «Коллекции и
точки») — заново энкодить весь текст не нужно.

Модель для token-level векторов — `fastembed`/`answerdotai/answerai-colbert-small-v1`,
130 МБ, считает эмбеддинги для всех 87 произведений на CPU за 1-2 секунды.

### Пример — коллекция с dense + colbert, prefetch + rerank

In [ ]:
from fastembed import LateInteractionTextEmbedding

# ColBERT-модель: вектор НЕ один на документ, а один на каждый токен (late interaction) -
# сравнение идёт токен-к-токену (MaxSim), а не по единственному усреднённому вектору
colbert_model = LateInteractionTextEmbedding(model_name="answerdotai/answerai-colbert-small-v1")

COLBERT_COLLECTION = PREFIX + "books_colbert"

texts = [b["title"] + ". " + b["text"] for b in BOOKS]
colbert_vectors = list(colbert_model.embed(texts))
COLBERT_DIM = colbert_vectors[0].shape[1]  # shape = (число токенов, размерность одного токен-вектора)
print("COLBERT_DIM:", COLBERT_DIM, "| токенов в первом документе:", colbert_vectors[0].shape[0])

# переиспользуем уже посчитанные dense-векторы из BOOKS_COLLECTION, а не считаем заново
existing = client.retrieve(BOOKS_COLLECTION, ids=[b["id"] for b in BOOKS], with_vectors=True)
dense_by_id = {p.id: p.vector for p in existing}

if client.collection_exists(COLBERT_COLLECTION):
    client.delete_collection(COLBERT_COLLECTION)

client.create_collection(
    collection_name=COLBERT_COLLECTION,
    vectors_config={
        "dense": VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        "colbert": VectorParams(
            size=COLBERT_DIM, distance=Distance.COSINE,
            # MAX_SIM - специальный компаратор для мультивекторов: score = сумма по запросным токенам
            # максимума косинуса с любым токеном документа (собственно, суть ColBERT)
            multivector_config=MultiVectorConfig(comparator=MultiVectorComparator.MAX_SIM),
        ),
    },
)

colbert_points = [
    PointStruct(
        id=b["id"],
        vector={"dense": dense_by_id[b["id"]], "colbert": cvec.tolist()},
        payload={k: v for k, v in b.items() if k != "id"},
    )
    for b, cvec in zip(BOOKS, colbert_vectors)
]
BATCH = 16
for i in range(0, len(colbert_points), BATCH):
    client.upsert(collection_name=COLBERT_COLLECTION, points=colbert_points[i:i + BATCH])
print("COLBERT_COLLECTION count:", client.count(COLBERT_COLLECTION).count)

query_text = "мрачная история об убийстве и мучительных муках совести"
dense_q = embed_query(query_text)
colbert_q = list(colbert_model.query_embed(query_text))[0].tolist()

# базовый dense-поиск - без всякого ColBERT, для сравнения "было/стало"
hits_plain = client.query_points(BOOKS_COLLECTION, query=dense_q, limit=5).points
# двухэтапный поиск: Prefetch отбирает 20 кандидатов по dense (дёшево), затем ColBERT честно
# пересчитывает порядок по MaxSim среди уже отобранных (дорого, но только на маленьком множестве)
hits_reranked = client.query_points(
    COLBERT_COLLECTION,
    prefetch=[Prefetch(query=dense_q, using="dense", limit=20)],
    query=colbert_q,
    using="colbert",
    limit=5,
).points

print("plain dense top-5:     ", [(h.id, round(h.score, 3)) for h in hits_plain])
print("colbert-reranked top-5:", [(h.id, round(h.score, 3)) for h in hits_reranked])


Реальный результат: plain dense top-5 — `[(10, 0.838), (32, 0.823), (41, 0.822), (3, 0.821), (17, 0.821)]`;
после ColBERT rerank top-5 — `[(40, 44.188), (17, 44.137), (54, 43.98), (41, 43.936), (18, 43.869)]`.
Обратите внимание на масштаб score — это уже не косинус (диапазон [-1, 1]), а сумма MaxSim по
токенам, поэтому числа выглядят непривычно большими и напрямую с dense-score не сравниваются.
Заметьте также: top-1 после rerank ("Леди Макбет Мценского уезда", тоже убийство, только другое) —
не обязательно "более правильный" ответ, чем исходный dense top-1 ("Преступление и наказание") —
токен-в-токен пересчёт честно меняет порядок на основании более детального сравнения, но не
гарантирует, что результат интуитивно понравится больше на маленькой коллекции.

## Задание 1

Для того же запроса `"мрачная история об убийстве и мучительных муках совести"`
(он же выше) сравните топ-1 до и после rerank: изменился ли он? Сохраните оба id в `top1_before` и `top1_after`,
и булев ответ в `top1_changed`.

**Используйте:** `client.query_points(...)` дважды — без `prefetch`/`using` (обычный dense-поиск по `BOOKS_COLLECTION`) и с `prefetch=[Prefetch(...)]`, `using="colbert"` (по `COLBERT_COLLECTION`).

In [ ]:
# ВАШ КОД ЗДЕСЬ
top1_before = ...
top1_after = ...
top1_changed = ...


In [ ]:
def check_ex1(top1_before, top1_after, top1_changed):
    assert (top1_before != top1_after) == top1_changed, "top1_changed не соответствует фактическому сравнению id"
    print("OK: top1_before={}, top1_after={}, top1_changed={}".format(top1_before, top1_after, top1_changed))
    print("ОТВЕТ ДЛЯ STEPIK:", top1_after)


check_ex1(top1_before, top1_after, top1_changed)


### Пример (без самопроверки) — MUVERA: приближённый первый этап вместо dense prefetch

Выше `Prefetch` брал кандидатов по обычному dense-вектору. MUVERA сжимает саму матрицу
token-векторов документа в один вектор фиксированной размерности (FDE) — его тоже можно
использовать для `Prefetch`, не привлекая отдельную dense-модель вообще:

In [ ]:
from fastembed.postprocess import Muvera

# muvera подбирается под конкретную модель - размерность токен-векторов берётся из неё автоматически
muvera = Muvera.from_multivector_model(colbert_model, k_sim=5, dim_proj=16, r_reps=20)
MUVERA_DIM = muvera.embedding_size
print("MUVERA_DIM:", MUVERA_DIM)

MUVERA_COLLECTION = PREFIX + "books_muvera"

if client.collection_exists(MUVERA_COLLECTION):
    client.delete_collection(MUVERA_COLLECTION)

client.create_collection(
    collection_name=MUVERA_COLLECTION,
    vectors_config={
        # MUVERA-приближение сравнивается обычным скалярным произведением, не косинусом -
        # это часть самого алгоритма, а не произвольный выбор
        "muvera": VectorParams(size=MUVERA_DIM, distance=Distance.DOT),
        "colbert": VectorParams(
            size=COLBERT_DIM, distance=Distance.COSINE,
            multivector_config=MultiVectorConfig(comparator=MultiVectorComparator.MAX_SIM),
        ),
    },
)

# process_document нормализует центры кластеров и заполняет пустые - для запроса ниже используем
# process_query, у которого этой нормализации нет, чтобы не исказить масштаб самого запроса
muvera_points = [
    PointStruct(
        id=b["id"],
        vector={"muvera": muvera.process_document(np.array(cvec)).tolist(), "colbert": cvec.tolist()},
        payload={k: v for k, v in b.items() if k != "id"},
    )
    for b, cvec in zip(BOOKS, colbert_vectors)
]
for i in range(0, len(muvera_points), BATCH):
    client.upsert(collection_name=MUVERA_COLLECTION, points=muvera_points[i:i + BATCH])
print("MUVERA_COLLECTION count:", client.count(MUVERA_COLLECTION).count)

query_fde = muvera.process_query(np.array(colbert_q)).tolist()

hits_muvera = client.query_points(
    collection_name=MUVERA_COLLECTION,
    prefetch=[Prefetch(query=query_fde, using="muvera", limit=20)],
    query=colbert_q,
    using="colbert",
    limit=5,
).points

print("MUVERA prefetch + ColBERT rerank top-5:", [(h.id, round(h.score, 3)) for h in hits_muvera])


Реальный результат: `[(17, 44.137), (28, 43.999), (70, 43.976), (87, 43.969), (41, 43.936)]` —
и это честный, не приукрашенный результат: настоящий топ-1 при полном переборе без prefetch —
id=40 ("Леди Макбет Мценского уезда", тот же, что и в примере выше) — в этот список вообще не
попал. MUVERA-приближение не включило его в свой пул из 20 кандидатов для `Prefetch`, а rerank
умеет только переупорядочивать то, что уже в пуле, а не находить кандидатов заново. Это и есть
цена приближения: на маленьком `limit` можно потерять то, что было бы в топе при честном переборе —
тот же компромисс, что и oversampling при квантовании, только для другого этапа поиска.